In [ ]:
# ============================================================
# Gaudinier et al. 2018
# Transcriptional regulation of nitrogen-associated metabolism and growth
# Faithful 5-output workflow
# 1) Figure1a PDF from repo
# 2) Figure1b PDF from repo
# 3) Figure4a PDF from repo
# 4) rebuilt Figure5a-style DEG barplot from SupplementalTable16
# 5) rebuilt Figure5bc-style feedback heatmaps from SupplementalTable16
# ============================================================

rm(list = ls())
graphics.off()
options(stringsAsFactors = FALSE, timeout = 600)

# ---------------------------
# 0. Packages
# ---------------------------
rm(list = ls())
graphics.off()

library(dplyr)
library(tidyr)
library(readxl)
library(ggplot2)
library(pheatmap)
library(gridExtra)
library(tibble)
library(stringr)
library(purrr)
library(RCy3)

# 1. 确认 Cytoscape 已经启动
cytoscapePing()

# 2. 路径
repo_dir <- "~/Gaudinier2018"
cy_dir   <- file.path(repo_dir, "CytoscapeFiles")
out_dir  <- file.path(repo_dir, "reexported_networks")
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

# 3. 查看有哪些 cys 文件
cys_files <- list.files(cy_dir, pattern = "\\.cys$", full.names = TRUE)
print(cys_files)

# 4. 一个稳妥的导出函数
export_one_session <- function(cys_path, out_prefix) {
  # 关闭当前 session，避免旧内容干扰
  try(closeSession(save.before.closing = FALSE), silent = TRUE)

  # 打开 session
  openSession(cys_path)

  # 看当前有哪些网络
  nets <- getNetworkList()
  print(nets)

  # 一般 session 打开后会自动出现对应 view
  # 如果有多个网络，就手动指定一个
  if (length(nets) >= 1) {
    setCurrentNetwork(nets[1])
  }

  # 导出 PDF
  exportImage(
    filename = file.path(out_dir, paste0(out_prefix, ".pdf")),
    type = "PDF"
  )

  # 同时导出 SVG，便于后期编辑
  exportImage(
    filename = file.path(out_dir, paste0(out_prefix, ".svg")),
    type = "SVG"
  )
}

# 5. 按文件名批量导出
for (f in cys_files) {
  nm <- tools::file_path_sans_ext(basename(f))
  export_one_session(f, nm)
}

list.files(out_dir, full.names = TRUE)

# ---------------------------
# 1. Directories
# ---------------------------
dir.create("gaudinier2018_data", showWarnings = FALSE, recursive = TRUE)
dir.create("gaudinier2018_figures", showWarnings = FALSE, recursive = TRUE)


old_pdfs <- list.files("gaudinier2018_figures", pattern = "\\.pdf$", full.names = TRUE)
if (length(old_pdfs) > 0) unlink(old_pdfs)

# ---------------------------
# 2. Robust downloader
# ---------------------------
download_github_file <- function(rel_path, out_file = NULL) {
  url <- paste0(
    "https://github.com/agaudinier/Gaudinier2018/raw/refs/heads/master/",
    gsub(" ", "%20", rel_path, fixed = TRUE)
  )

  if (is.null(out_file)) {
    out_file <- file.path("gaudinier2018_data", basename(rel_path))
  }

  if (file.exists(out_file) && isTRUE(file.info(out_file)$size > 1000)) {
    return(normalizePath(out_file))
  }

  ok <- FALSE

  # 1) wget
  if (!ok && nzchar(Sys.which("wget"))) {
    status <- suppressWarnings(
      system2("wget", args = c("-q", "-O", out_file, url))
    )
    ok <- identical(status, 0L) &&
      file.exists(out_file) &&
      isTRUE(file.info(out_file)$size > 1000)
  }

  # 2) curl
  if (!ok && nzchar(Sys.which("curl"))) {
    status <- suppressWarnings(
      system2("curl", args = c("-L", "-o", out_file, url))
    )
    ok <- identical(status, 0L) &&
      file.exists(out_file) &&
      isTRUE(file.info(out_file)$size > 1000)
  }

  # 3) download.file(libcurl)
  if (!ok) {
    try(
      download.file(
        url = url,
        destfile = out_file,
        mode = "wb",
        method = "libcurl",
        quiet = TRUE
      ),
      silent = TRUE
    )
    ok <- file.exists(out_file) && isTRUE(file.info(out_file)$size > 1000)
  }

  if (!ok) {
    stop("Download failed: ", rel_path)
  }

  normalizePath(out_file)
}

# ---------------------------
# 3. Download exact repo PDFs + Supplemental Table 16
# ---------------------------
repo_pdf_files <- c(
  fig1a = "NetworkPDFs/Figure1a.pdf",
  fig1b = "NetworkPDFs/Figure1b.pdf",
  fig4a = "NetworkPDFs/Figure4a.pdf"
)

pdf_paths <- purrr::imap_chr(
  repo_pdf_files,
  ~ download_github_file(.x, file.path("gaudinier2018_data", paste0(.y, ".pdf")))
)

tbl16_path <- download_github_file(
  "Supplemental Tables/SupplementalTable16.xls",
  file.path("gaudinier2018_data", "SupplementalTable16.xls")
)


file.copy(pdf_paths["fig1a"], "gaudinier2018_figures/01_Figure1a_exact_from_repo.pdf", overwrite = TRUE)
file.copy(pdf_paths["fig1b"], "gaudinier2018_figures/02_Figure1b_exact_from_repo.pdf", overwrite = TRUE)
file.copy(pdf_paths["fig4a"], "gaudinier2018_figures/03_Figure4a_exact_from_repo.pdf", overwrite = TRUE)

# ---------------------------
# 4. Read SupplementalTable16 exactly
# ---------------------------
read_t16 <- function(sheet_name) {
  readxl::read_excel(
    tbl16_path,
    sheet = sheet_name,
    col_types = "text",
    na = c("", "NA", "N/A")
  )
}

t16_all <- read_t16("a. Significant DE Genes")
t16_ynm <- read_t16("b. YNM DE Genes")

# ---------------------------
# 5. Dataset definitions from Table16
# ---------------------------
dataset_defs <- list(
  tga1tga4 = list(
    logfc = c("tga1tga4.tga1tga4.logFC", "tga1tga4.KNO3.logFC"),
    padj  = c("tga1tga4.adj.P.Val")
  ),
  chl1 = list(
    logfc = c("chl1.9.logFC", "chl1.12.logFC", "chl1.5.logFC"),
    padj  = c("chl1.9.adj.P.Val", "chl1.12.adj.P.Val", "chl1.5.adj.P.Val")
  ),
  anr1 = list(
    logfc = c("anr1.logFC"),
    padj  = c("anr1.adj.P.Val")
  ),
  nia1nia2 = list(
    logfc = c("nia1nia2.logFC"),
    padj  = c("nia1nia2.adj.P.Val")
  ),
  glu1 = list(
    logfc = c("glu1_root.logFC", "glu1_leaf.logFC"),
    padj  = c("glu1_root.adj.P.Val", "glu1_leaf.adj.P.Val")
  ),
  bzip1 = list(
    logfc = c("bzip1.logFC"),
    padj  = c("bzip1.adj.P.Val")
  ),
  nrt2_4 = list(
    logfc = c("nrt2.4.logFC"),
    padj  = c("nrt2.4.adj.P.Val")
  ),
  nin7_1 = list(
    logfc = c("nin7.1.logFC"),
    padj  = c("nin7.1.adj.P.Val")
  ),
  nlp6_sup = list(
    logfc = c("nlp6_sup14.logFC", "nlp6_sup7.logFC"),
    padj  = c("nlp6_sup14.adj.P.Val", "nlp6_sup7.adj.P.Val")
  ),
  nlp6nlp7 = list(
    logfc = c("nlp6nlp7_t10.logFC", "nlp6nlp7_t20.logFC"),
    padj  = c("nlp6nlp7.adj.P.Val")
  ),
  nlp7_1 = list(
    logfc = c("nlp7.1_t10.logFC", "nlp7.1_t20.logFC"),
    padj  = c("nlp7.1.adj.P.Val")
  ),
  nlp7_3 = list(
    logfc = c("nlp7.3_t10.logFC", "nlp7.3_t20.logFC"),
    padj  = c("nlp7.3.adj.P.Val")
  ),
  gdh_triple = list(
    logfc = c("gdh1gdh2gdh3.logFC"),
    padj  = c("gdh1gdh2gdh3.adj.P.Val")
  )
)

dataset_order <- c(
  "tga1tga4", "chl1", "anr1", "nia1nia2", "glu1",
  "bzip1", "nrt2_4", "nin7_1", "nlp6_sup",
  "nlp6nlp7", "nlp7_1", "nlp7_3", "gdh_triple"
)

# ---------------------------
# 6. Helpers
# ---------------------------
to_num <- function(x) suppressWarnings(as.numeric(x))

check_required_columns <- function(df, dataset_defs) {
  required_cols <- unique(c(
    "AGI", "Gene Name",
    unlist(lapply(dataset_defs, function(z) z$logfc)),
    unlist(lapply(dataset_defs, function(z) z$padj))
  ))
  miss <- setdiff(required_cols, names(df))
  if (length(miss) > 0) {
    stop("Missing columns in Table16: ", paste(miss, collapse = ", "))
  }
}

as_num_matrix <- function(df, cols) {
  if (length(cols) == 1) {
    m <- matrix(to_num(df[[cols]]), ncol = 1)
    colnames(m) <- cols
    return(m)
  }
  m <- sapply(cols, function(cc) to_num(df[[cc]]))
  if (is.vector(m)) m <- matrix(m, ncol = 1)
  m
}

row_mean_na <- function(m) {
  apply(m, 1, function(z) {
    z <- z[!is.na(z)]
    if (length(z) == 0) return(NA_real_)
    mean(z)
  })
}

row_min_na <- function(m) {
  apply(m, 1, function(z) {
    z <- z[!is.na(z)]
    if (length(z) == 0) return(NA_real_)
    min(z)
  })
}

build_long_table16 <- function(df, dataset_defs) {
  check_required_columns(df, dataset_defs)

  long_list <- purrr::imap(dataset_defs, function(def, nm) {
    logfc_m <- as_num_matrix(df, def$logfc)
    padj_m  <- as_num_matrix(df, def$padj)

    tibble::tibble(
      agi = df[["AGI"]],
      gene = ifelse(
        is.na(df[["Gene Name"]]) | df[["Gene Name"]] == "",
        df[["AGI"]],
        df[["Gene Name"]]
      ),
      dataset = nm,
      logFC = row_mean_na(logfc_m),
      adj.P.Val = row_min_na(padj_m)
    )
  })

  dplyr::bind_rows(long_list)
}

make_unique_gene_labels <- function(gene_vec, agi_vec) {
  out <- gene_vec
  dup <- duplicated(out) | duplicated(out, fromLast = TRUE)
  out[dup] <- paste0(out[dup], " (", agi_vec[dup], ")")
  out
}

build_heatmap_matrix <- function(long_df, genes_keep, dataset_order) {
  tmp <- long_df %>%
    filter(gene %in% genes_keep) %>%
    mutate(plot_value = ifelse(!is.na(adj.P.Val) & adj.P.Val < 0.05, logFC, 0)) %>%
    select(agi, gene, dataset, plot_value)

  if (nrow(tmp) == 0) stop("No matching genes found for heatmap.")

  label_df <- tmp %>%
    distinct(agi, gene) %>%
    mutate(gene_label = make_unique_gene_labels(gene, agi))

  tmp <- tmp %>%
    left_join(label_df, by = c("agi", "gene")) %>%
    select(gene_label, dataset, plot_value)

  wide <- tmp %>%
    tidyr::pivot_wider(names_from = dataset, values_from = plot_value, values_fill = 0)

  missing_ds <- setdiff(dataset_order, names(wide))
  for (ds in missing_ds) wide[[ds]] <- 0

  wide <- wide %>%
    select(gene_label, all_of(dataset_order))

  mat <- wide %>%
    tibble::column_to_rownames("gene_label") %>%
    as.matrix()

  storage.mode(mat) <- "numeric"

  # 去掉全 0 行
  keep <- rowSums(abs(mat) > 0, na.rm = TRUE) > 0
  mat <- mat[keep, , drop = FALSE]

  # 按显著数据集个数排序
  if (nrow(mat) > 1) {
    mat <- mat[order(rowSums(abs(mat) > 0, na.rm = TRUE), decreasing = TRUE), , drop = FALSE]
  }

  mat
}

make_diverging_breaks <- function(mat, n = 101) {
  lim <- max(abs(mat), na.rm = TRUE)
  if (!is.finite(lim) || lim < 1e-6) lim <- 1
  seq(-lim, lim, length.out = n)
}

make_heatmap_grob <- function(mat, title_txt) {
  brks <- make_diverging_breaks(mat, 101)
  cols <- grDevices::colorRampPalette(c("#2166AC", "white", "#B2182B"))(100)

  p <- pheatmap::pheatmap(
    mat,
    color = cols,
    breaks = brks,
    cluster_rows = nrow(mat) > 1,
    cluster_cols = FALSE,
    border_color = "grey90",
    fontsize = 10,
    fontsize_row = 8,
    fontsize_col = 10,
    angle_col = 45,
    main = title_txt,
    silent = TRUE
  )

  p$gtable
}

safe_pdf <- function(file, width, height, expr) {
  pdf(file, width = width, height = height, useDingbats = FALSE)
  on.exit(dev.off(), add = TRUE)
  force(expr)
}

# ---------------------------
# 7. Build long tables directly from Table16
# ---------------------------
all_long <- build_long_table16(t16_all, dataset_defs)
ynm_long <- build_long_table16(t16_ynm, dataset_defs)

# ---------------------------
# 8. Figure5a-style barplot
# ---------------------------
count_all <- all_long %>%
  group_by(dataset) %>%
  summarise(n_all = sum(!is.na(adj.P.Val) & adj.P.Val < 0.05), .groups = "drop")

count_ynm <- ynm_long %>%
  group_by(dataset) %>%
  summarise(n_ynm = sum(!is.na(adj.P.Val) & adj.P.Val < 0.05), .groups = "drop")

count_df <- full_join(count_all, count_ynm, by = "dataset") %>%
  mutate(
    n_all = ifelse(is.na(n_all), 0, n_all),
    n_ynm = ifelse(is.na(n_ynm), 0, n_ynm)
  ) %>%
  mutate(dataset = factor(dataset, levels = dataset_order)) %>%
  arrange(desc(n_all))

p5a <- ggplot(count_df, aes(x = dataset)) +
  geom_col(aes(y = n_all), fill = "grey85", width = 0.75) +
  geom_col(aes(y = n_ynm), fill = "grey35", width = 0.45) +
  coord_flip() +
  theme_bw(base_size = 12) +
  labs(
    title = "Figure 5a-style: differentially expressed genes by perturbation dataset",
    subtitle = "Light grey = all significant DE genes; dark grey = YNM DE genes",
    x = NULL,
    y = "Gene count"
  )

ggsave(
  filename = "gaudinier2018_figures/04_Figure5a_rebuilt_from_Table16.pdf",
  plot = p5a,
  width = 8,
  height = 6,
  device = cairo_pdf
)

# ---------------------------
# 9. Figure5bc-style heatmaps
# ---------------------------
regulator_genes <- c(
  "ANR1", "NLP7", "NLP6", "LBD37", "LBD38", "LBD39",
  "TGA1", "ARF18", "ARF9", "ARID5", "ATAF1",
  "BBX16", "ERF107", "HAT22", "HMGB15",
  "LBD4", "LHY", "MYB29", "RAV2"
)

metabolism_genes <- c(
  "ASN1", "ASN2", "CIPK23", "CIPK8", "GDH1",
  "NIA1", "NIA2", "NIR1", "NPF6.3", "NPF7.3",
  "NRT2.1", "NRT2.4"
)

reg_mat <- build_heatmap_matrix(ynm_long, regulator_genes, dataset_order)
met_mat <- build_heatmap_matrix(ynm_long, metabolism_genes, dataset_order)

reg_grob <- make_heatmap_grob(reg_mat, "Figure 5b-style: transcriptional regulators")
met_grob <- make_heatmap_grob(met_mat, "Figure 5c-style: nitrogen metabolism / transport genes")

safe_pdf(
  file = "gaudinier2018_figures/05_Figure5bc_feedback_heatmaps_rebuilt_from_Table16.pdf",
  width = 15,
  height = max(8, 0.22 * max(nrow(reg_mat), nrow(met_mat)) + 4),
  expr = {
    gridExtra::grid.arrange(reg_grob, met_grob, ncol = 2)
  }
)

# ---------------------------
# 10. Report outputs
# ---------------------------
out_files <- list.files("gaudinier2018_figures", pattern = "\\.pdf$", full.names = TRUE)
message("Done. Generated files:")
print(out_files)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.3.3”

Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




[1] TRUE

[1] TRUE

[1] TRUE

Done. Generated files:



[1] "gaudinier2018_figures/01_Figure1a_exact_from_repo.pdf"                        
[2] "gaudinier2018_figures/02_Figure1b_exact_from_repo.pdf"                        
[3] "gaudinier2018_figures/03_Figure4a_exact_from_repo.pdf"                        
[4] "gaudinier2018_figures/04_Figure5a_rebuilt_from_Table16.pdf"                   
[5] "gaudinier2018_figures/05_Figure5bc_feedback_heatmaps_rebuilt_from_Table16.pdf"
